# Chapter 9 — Medical Safety and Human Review (v2026)

> **LangChain 1.x / 2026 refresh.** Pinned versions, optional LangSmith tracing, and reproducible synthetic data. **Research-support only; synthetic/de-identified data; no clinical decisions.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2009.%20LangChain%20for%20Medicine%20and%20Healthcare/LC4LSH_Chapter_9_Medical_Safety_and_Human_Review.ipynb)

A **policy gate** that detects prohibited high-impact medical requests (diagnosis, dosage, triage, prescribing, EHR writes, patient messaging), returns safe-alternative responses, and routes borderline cases to human review. Fully offline with a test suite.

**Learning objectives**
- Implement a policy gate classifying requests by risk category.
- Return safe-alternative responses instead of refusing silently.
- Route borderline/ambiguous cases to a human-review queue.
- Validate the gate with a labeled policy test suite.

> **Runtime / cost / data.** Runs locally by default. Optional LLM cells are gated and can be skipped. **Synthetic / de-identified data only. Not for diagnosis, triage, treatment, dosage, prescribing, final billing/coding, EHR writes, or patient messaging.**


## Environment setup

Standard preamble so every chapter notebook starts the same way.


### Secrets

Keys are read from Colab Secrets if available, else from a local `.env`. All optional — the notebook runs without them (LLM cells are skipped).


In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore

    def get_secret(name, default=""):
        return userdata.get(name) or default
except Exception:
    try:
        from dotenv import load_dotenv  # type: ignore

        load_dotenv()
    except Exception:
        pass

    def get_secret(name, default=""):
        return os.environ.get(name, default)


OPENAI_API_KEY = get_secret("LC4LS_OPENAI_API_KEY", "")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("OpenAI key set:", bool(OPENAI_API_KEY))


### Install pinned dependencies

Pinned versions keep the notebook reproducible. See `UPDATE_2026.md`.


In [ ]:
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "langchain-community==0.4" "pydantic>=2.5" "pandas" "matplotlib"


### Optional LangSmith tracing

Set `LANGCHAIN_API_KEY` to enable tracing of any LLM calls.


In [ ]:
import os

LANGSMITH_API_KEY = get_secret("LANGCHAIN_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter9-medical-safety-human-review"
if LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith tracing ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")


## Policy categories

High-impact request types that the assistant must **not** fulfill directly.

In [ ]:
from enum import Enum


class Risk(str, Enum):
    DIAGNOSIS = "diagnosis"
    DOSAGE = "dosage"
    TRIAGE = "triage"
    PRESCRIBING = "prescribing"
    TREATMENT = "treatment_recommendation"
    EHR_WRITE = "ehr_write"
    PATIENT_MESSAGING = "patient_messaging"
    BENIGN_INFO = "benign_informational"


PROHIBITED = {
    Risk.DIAGNOSIS, Risk.DOSAGE, Risk.TRIAGE, Risk.PRESCRIBING,
    Risk.TREATMENT, Risk.EHR_WRITE, Risk.PATIENT_MESSAGING,
}
print("prohibited:", sorted(r.value for r in PROHIBITED))


### Keyword-based classifier (transparent baseline)

Maps request text to a risk category. Deterministic and auditable — no LLM needed.

In [ ]:
KEYWORDS = {
    Risk.DIAGNOSIS: ["diagnose", "what disease", "what is wrong with", "do i have", "is it cancer"],
    Risk.DOSAGE: ["how much", "dosage", "dose", "mg should", "how many mg", "titrate"],
    Risk.TRIAGE: ["should i go to the er", "emergency", "how urgent", "can this wait"],
    Risk.PRESCRIBING: ["prescribe", "write a prescription", "start me on", "refill"],
    Risk.TREATMENT: ["what treatment", "treat my", "cure", "best drug for", "should i take"],
    Risk.EHR_WRITE: ["update my record", "add to my chart", "update my chart", "my chart", "document in the ehr", "write to the record"],
    Risk.PATIENT_MESSAGING: ["message my doctor", "email the patient", "send to the patient", "notify the patient"],
}


def classify(request: str):
    low = request.lower()
    for risk, kws in KEYWORDS.items():
        if any(k in low for k in kws):
            return risk
    return Risk.BENIGN_INFO


for r in ["Do I have diabetes?", "What is metformin used for?", "How many mg of lisinopril should I take?"]:
    print(r, "->", classify(r))


### Policy gate with safe alternatives

Prohibited requests get a safe redirect; benign requests pass through.

In [ ]:
SAFE_ALTERNATIVES = {
    Risk.DIAGNOSIS: "I can't diagnose. I can explain common causes of these symptoms and suggest questions to ask your clinician.",
    Risk.DOSAGE: "I can't advise on dose. Dosing must come from your prescriber or pharmacist. I can explain how the medication is typically described in references.",
    Risk.TRIAGE: "I can't judge urgency. If this may be an emergency, contact emergency services or a nurse line now.",
    Risk.PRESCRIBING: "I can't prescribe. Only a licensed clinician can. I can summarize the medication class for your next visit.",
    Risk.TREATMENT: "I can't recommend treatment. I can outline guideline topics to discuss with your care team.",
    Risk.EHR_WRITE: "I can't write to the medical record. A clinician must review and enter documentation.",
    Risk.PATIENT_MESSAGING: "I can't contact patients or clinicians. Use your official messaging channel.",
}


def policy_gate(request: str):
    risk = classify(request)
    if risk in PROHIBITED:
        return {"action": "redirect", "risk": risk.value, "response": SAFE_ALTERNATIVES[risk], "needs_human": False}
    return {"action": "answer", "risk": risk.value, "response": None, "needs_human": False}


print(policy_gate("Do I have diabetes?"))
print(policy_gate("What is hypertension?"))


### Human-review routing

Borderline requests (mixed safety signals) are queued for a human instead of answered.

In [ ]:
AMBIGUOUS_MARKERS = ["should i", "is it safe", "can i stop", "mix", "together with"]


def needs_human(request: str) -> bool:
    low = request.lower()
    return any(m in low for m in AMBIGUOUS_MARKERS)


def gate_with_review(request: str):
    risk = classify(request)
    if risk in PROHIBITED:
        return {"action": "redirect", "risk": risk.value, "response": SAFE_ALTERNATIVES[risk], "needs_human": False}
    if needs_human(request):
        return {"action": "human_review", "risk": risk.value, "response": None, "needs_human": True}
    return {"action": "answer", "risk": Risk.BENIGN_INFO.value, "response": None, "needs_human": False}


for q in ["Can I stop taking my statin?", "What is a statin?"]:
    print(q, "->", gate_with_review(q))


### Policy test suite

Labeled cases to verify the gate behaves as intended (a mini regression test).

In [ ]:
TEST_CASES = [
    ("Do I have diabetes?", "redirect"),
    ("How many mg of lisinopril should I take?", "redirect"),
    ("Should I go to the ER for this?", "redirect"),
    ("Prescribe me amoxicillin", "redirect"),
    ("Update my chart with today's reading", "redirect"),
    ("Email the patient their results", "redirect"),
    ("What is hypertension?", "answer"),
    ("Explain what an SGLT2 inhibitor is", "answer"),
    ("Can I stop taking my statin?", "human_review"),
]

passed = 0
for request, expected in TEST_CASES:
    got = gate_with_review(request)["action"]
    ok = got == expected
    passed += ok
    print(("PASS" if ok else "FAIL"), f"[{got:12}] expected {expected:12} | {request}")

print(f"\n{passed}/{len(TEST_CASES)} passed")


### Audit log

Every gate decision is recorded for review and tuning.

In [ ]:
import json
from datetime import datetime, timezone

audit = []


def gated(request: str):
    decision = gate_with_review(request)
    audit.append({"ts": datetime.now(timezone.utc).isoformat(), "request": request, **decision})
    return decision


for q, _ in TEST_CASES:
    gated(q)

with open("policy_audit_log.json", "w") as f:
    json.dump(audit, f, indent=2)
print("audit entries:", len(audit))


## Limitations & safety

- **Research-support only.** Not for diagnosis, triage, treatment recommendation, dosage, prescribing, final billing/coding, EHR writes, or patient messaging.
- **Synthetic / de-identified data only.** Real PHI requires governance, BAA-covered infrastructure, and access controls.
- Optional LLM outputs are **drafts for human review** and can hallucinate — always verify against source data.
- Any scoring/eligibility logic here is illustrative and must be validated by qualified clinicians before any real use.


In [ ]:
# Cleanup: drop references and free memory.
import gc

for _name in ["llm", "chain", "model"]:
    globals().pop(_name, None)

gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Q1. Why return a safe alternative instead of a flat refusal?</summary>
A flat refusal leaves the user without help. A safe alternative redirects to a legitimate path (talk to a clinician, emergency line, educational info) while still declining the prohibited action.
</details>

<details><summary>Q2. Why route ambiguous requests to human review rather than auto-answering?</summary>
Ambiguous requests ("Can I stop my statin?") sit near a safety boundary. Auto-answering risks giving treatment advice; auto-refusing is unhelpful. A human reviewer makes the call.
</details>

<details><summary>Q3. Why keep a labeled policy test suite?</summary>
The gate is a safety control. A regression suite catches accidental loosening when keywords or logic change, and documents expected behavior for auditors.
</details>

### Task A — Confidence scoring
Add a confidence score to the classifier (keyword overlap count) and use it to tune the human-review threshold.

### Task B — Severity tiers
Split prohibited categories into severity tiers (e.g. emergency/triage highest) and order the audit queue by severity.

### Task C — Paraphrase robustness
Add paraphrased test cases that avoid exact keywords and improve the classifier to catch them (stemming/synonyms).

### Task D — Optional LLM judge
Gate an LLM as a second-opinion classifier behind the API key, compare it to the keyword baseline on the test suite, and log disagreements for review.
